# Лаборатори 4: Нэгдүгээр эрэмбийн логик
## Prolog ба pyDatalog ашиглан гэр бүлийн мэдлэгийн сан байгуулах

**Зорилго:**
Энэхүү лабораторийн хичээлээр оюутан нэгдүгээр эрэмбийн логикийн баримт (fact), дүрэм (rule), асуулт (query) гэсэн гурван ойлголтыг практикт ашиглаж, pyDatalog болон Prolog хэлээр гэр бүлийн харилцааны мэдлэгийн сан байгуулан, түүнд асуулт тавьж хариулт авах чадвартай болно.

Алхам 1–7 ба нэмэлт даалгаврыг энэ notebook-д pyDatalog-оор, Алхам 8-ыг `family.pl` файлд Prolog-оор хийв. Алхам 7-ийн хүснэгт, Алхам 4 ба 8-ын үгээр хариулах асуултууд notebook-ийн төгсгөлд байна.

## Алхам 1: Орчин бэлтгэх

pyDatalog-ийг суулгаад 4.5-ын жижиг жишээг (bird, flies) ажиллуулна.

In [1]:
%pip install -q --disable-pip-version-check pyDatalog

Note: you may need to restart the kernel to use updated packages.


**Шалгах:** `flies(X)` нь sparrow, tweety гэсэн 2 утга буцаана. Cell-ийг дахин ажиллуулахыг давталтаар 2 удаа ажиллуулж шалгав. Эхэнд `pyDatalog.clear()` байгаа тул 2 дахь удаад ч 2 утга гарах ёстой.

In [2]:
from pyDatalog import pyDatalog

for run in range(2):
    pyDatalog.clear()
    pyDatalog.create_terms('bird, flies, X')
    +bird('tweety')
    +bird('sparrow')
    flies(X) <= bird(X)
    print(f'{run + 1}-р удаа:')
    print(flies(X))
    print()

1-р удаа:
X      
-------
sparrow
tweety 

2-р удаа:
X      
-------
sparrow
tweety 



Хоёр удаад хоёулаа 2 утга гарсан. Жишээ баримтыг `pyDatalog.clear()`-ээр арилгаад дараагийн алхмуудын нэрсийг нэг cell-д зарлав.

In [3]:
pyDatalog.clear()
pyDatalog.create_terms('male, female, parent, person, has_child, childless, '
                       'father, mother, son, daughter, grandparent, sibling, brother, '
                       'sister, uncle, aunt, cousin, ancestor, descendant, great_grandparent, '
                       'X, Y, Z, P, Q');

## Алхам 2: Гэр бүлийн мэдлэгийн сан (баримт)

15 хүн, 4 үе. `male` 8, `female` 7 баримттай. `parent` баримтыг гараар 18 мөр бичихгүйн тулд гэр бүл бүрийг (эцэг эх хос, хүүхдүүд) жагсаагаад давталтаар нэмэв. Эцэг, эх хоёр тус тусдаа баримт болно.

`count(q)` нь хариултын тоог, `names(q)` нь эхний хувьсагчийн утгуудыг давхардалгүй эрэмбэлж буцаана. `pyDatalog.ask` хариу олдохгүй үед None буцаадаг тул 0 болон хоосон жагсаалт болгосон.

In [4]:
for p in ['bataa', 'bold', 'tomor', 'dorj', 'zolboo', 'munkh', 'temuulen', 'bat']:
    +male(p)

for p in ['saraa', 'oyuna', 'tsetseg', 'ganaa', 'anar', 'naraa', 'enkhee']:
    +female(p)

families = [
    (['bataa', 'saraa'], ['bold', 'oyuna', 'tomor']),
    (['bold', 'tsetseg'], ['anar', 'zolboo']),
    (['dorj', 'oyuna'], ['munkh', 'naraa']),
    (['tomor', 'ganaa'], ['enkhee']),
    (['bat', 'anar'], ['temuulen']),
]

for parents, children in families:
    for p in parents:
        for c in children:
            +parent(p, c)


def count(q):
    r = pyDatalog.ask(q)
    return 0 if r is None else len(r.answers)


def names(q):
    r = pyDatalog.ask(q)
    return [] if r is None else sorted({a[0] for a in r.answers})

**Шалгах:** `parent` 18, `male` 8, `female` 7. `parent('bataa', X)` нь bold, oyuna, tomor.

In [5]:
for q, expected in [('parent(X, Y)', 18), ('male(X)', 8), ('female(X)', 7)]:
    print(f'count("{q}") = {count(q)}  (хүлээгдэж буй {expected})')
print()
print("parent('bataa', X):", names("parent('bataa', X)"))

count("parent(X, Y)") = 18  (хүлээгдэж буй 18)
count("male(X)") = 8  (хүлээгдэж буй 8)
count("female(X)") = 7  (хүлээгдэж буй 7)

parent('bataa', X): ['bold', 'oyuna', 'tomor']


## Алхам 3: Энгийн дүрэм

Баримтад `+`, дүрэмд `<=` хэрэглэнэ. Дүрмийн толгой (зүүн тал) нь дүгнэлт, бие (баруун тал) нь нөхцөл. `person`-д 2 дүрэм бичсэн нь "эсвэл" гэсэн утгатай. Дүрэм бүрийн дээр утгыг ба FOL томъёог comment болгон бичив.

In [6]:
father(X, Y) <= parent(X, Y) & male(X)

mother(X, Y) <= parent(X, Y) & female(X)

son(X, Y) <= parent(Y, X) & male(X)

daughter(X, Y) <= parent(Y, X) & female(X)

person(X) <= male(X)

person(X) <= female(X);

**Шалгах:** `father(X, 'anar')` нь bold, `mother(X, 'bold')` нь saraa, `son(X, 'oyuna')` нь munkh, `daughter(X, 'oyuna')` нь naraa. `father`, `mother` тус бүр 9, `person` 15.

In [7]:
for q, expected in [("father(X, 'anar')", ['bold']), ("mother(X, 'bold')", ['saraa']),
                    ("son(X, 'oyuna')", ['munkh']), ("daughter(X, 'oyuna')", ['naraa'])]:
    print(f'{q:22} {names(q)}  (хүлээгдэж буй {expected})')
print()
for q, expected in [('father(X, Y)', 9), ('mother(X, Y)', 9), ('person(X)', 15)]:
    print(f'count("{q}") = {count(q)}  (хүлээгдэж буй {expected})')

father(X, 'anar')      ['bold']  (хүлээгдэж буй ['bold'])
mother(X, 'bold')      ['saraa']  (хүлээгдэж буй ['saraa'])
son(X, 'oyuna')        ['munkh']  (хүлээгдэж буй ['munkh'])
daughter(X, 'oyuna')   ['naraa']  (хүлээгдэж буй ['naraa'])

count("father(X, Y)") = 9  (хүлээгдэж буй 9)
count("mother(X, Y)") = 9  (хүлээгдэж буй 9)
count("person(X)") = 15  (хүлээгдэж буй 15)


## Алхам 4: Өвөө эмээ ба ах дүү

`grandparent`-д дунд хүн Z, `sibling`-д нийтлэг эцэг эх P гэсэн нэмэлт хувьсагч хэрэгтэй. `sibling`-д `X != Y` заавал байна.

**grandparent дахь Z:** Z нь зөвхөн дүрмийн биед байна. Дүрэм бүхэлдээ ∀z-тэй ч z нь толгойд байхгүй үед ∀z (A(z) → B) нь (∃z A(z)) → B-тэй тэнцүү. Тиймээс биеийн дотор Z нь **∃**: "X-ийн хүүхэд, Y-ийн эцэг эх болох ядаж нэг Z хүн байвал хангалттай" гэсэн утга.

In [8]:
grandparent(X, Y) <= parent(X, Z) & parent(Z, Y)

sibling(X, Y) <= parent(P, X) & parent(P, Y) & (X != Y);

**Шалгах:** `grandparent(X, 'temuulen')` нь bold, tsetseg. `grandparent` 12. `sibling(X, 'munkh')` нь naraa, `sibling(X, 'bold')` нь oyuna, tomor. `sibling` 10 (эрэмбэлсэн хос).

`X != Y`-гүй хувилбарыг тусад нь `sibling_all` нэрээр бичиж тоолов. 19 байх ёстой.

In [9]:
for q, expected in [("grandparent(X, 'temuulen')", ['bold', 'tsetseg']), ("sibling(X, 'munkh')", ['naraa']),
                    ("sibling(X, 'bold')", ['oyuna', 'tomor'])]:
    print(f'{q:27} {names(q)}  (хүлээгдэж буй {expected})')
print()
for q, expected in [('grandparent(X, Y)', 12), ('sibling(X, Y)', 10)]:
    print(f'count("{q}") = {count(q)}  (хүлээгдэж буй {expected})')

pyDatalog.create_terms('sibling_all')
sibling_all(X, Y) <= parent(P, X) & parent(P, Y)

print(f'count("sibling_all(X, Y)") = {count("sibling_all(X, Y)")}  (хүлээгдэж буй 19)')
extra = set(pyDatalog.ask('sibling_all(X, Y)').answers) - set(pyDatalog.ask('sibling(X, Y)').answers)
print('Нэмэгдсэн', len(extra), 'мөр:', sorted(extra))

grandparent(X, 'temuulen')  ['bold', 'tsetseg']  (хүлээгдэж буй ['bold', 'tsetseg'])
sibling(X, 'munkh')         ['naraa']  (хүлээгдэж буй ['naraa'])
sibling(X, 'bold')          ['oyuna', 'tomor']  (хүлээгдэж буй ['oyuna', 'tomor'])

count("grandparent(X, Y)") = 12  (хүлээгдэж буй 12)
count("sibling(X, Y)") = 10  (хүлээгдэж буй 10)
count("sibling_all(X, Y)") = 19  (хүлээгдэж буй 19)
Нэмэгдсэн 9 мөр: [('anar', 'anar'), ('bold', 'bold'), ('enkhee', 'enkhee'), ('munkh', 'munkh'), ('naraa', 'naraa'), ('oyuna', 'oyuna'), ('temuulen', 'temuulen'), ('tomor', 'tomor'), ('zolboo', 'zolboo')]


Нэмэгдсэн 9 мөр бүгд (X, X) буюу хүн өөрийнхөө ах дүү болсон. Тайлбар нь төгсгөлийн "Хариултууд" хэсэгт.

## Алхам 5: Ураг төрлийн дүрмүүд

Шинэ дүрэм өмнөх дүрмийг дахин ашиглана: `brother`, `sister` нь `sibling` дээр, `uncle`, `aunt` нь `brother`, `sister` дээр, `cousin` нь `sibling` дээр тулгуурлана. Энд `uncle`, `aunt` нь зөвхөн цусан төрөл.

In [10]:
brother(X, Y) <= sibling(X, Y) & male(X)

sister(X, Y) <= sibling(X, Y) & female(X)

uncle(X, Y) <= parent(P, Y) & brother(X, P)

aunt(X, Y) <= parent(P, Y) & sister(X, P)

cousin(X, Y) <= parent(P, X) & parent(Q, Y) & sibling(P, Q);

**Шалгах:** `brother(X, 'oyuna')` нь bold, tomor. `sister(X, 'zolboo')` нь anar. `uncle(X, 'anar')` нь tomor, `aunt(X, 'anar')` нь oyuna, `uncle(X, 'temuulen')` нь zolboo. `cousin(X, 'enkhee')` 4 хүн, `cousin(X, 'anar')` нь enkhee, munkh, naraa. `cousin` 16.

In [11]:
for q, expected in [("brother(X, 'oyuna')", ['bold', 'tomor']), ("sister(X, 'zolboo')", ['anar']),
                    ("uncle(X, 'anar')", ['tomor']), ("aunt(X, 'anar')", ['oyuna']),
                    ("uncle(X, 'temuulen')", ['zolboo']),
                    ("cousin(X, 'enkhee')", ['anar', 'munkh', 'naraa', 'zolboo']),
                    ("cousin(X, 'anar')", ['enkhee', 'munkh', 'naraa'])]:
    print(f'{q:21} {names(q)}  (хүлээгдэж буй {expected})')
print()
print(f'count("cousin(X, Y)") = {count("cousin(X, Y)")}  (хүлээгдэж буй 16)')

brother(X, 'oyuna')   ['bold', 'tomor']  (хүлээгдэж буй ['bold', 'tomor'])
sister(X, 'zolboo')   ['anar']  (хүлээгдэж буй ['anar'])
uncle(X, 'anar')      ['tomor']  (хүлээгдэж буй ['tomor'])
aunt(X, 'anar')       ['oyuna']  (хүлээгдэж буй ['oyuna'])
uncle(X, 'temuulen')  ['zolboo']  (хүлээгдэж буй ['zolboo'])
cousin(X, 'enkhee')   ['anar', 'munkh', 'naraa', 'zolboo']  (хүлээгдэж буй ['anar', 'munkh', 'naraa', 'zolboo'])
cousin(X, 'anar')     ['enkhee', 'munkh', 'naraa']  (хүлээгдэж буй ['enkhee', 'munkh', 'naraa'])

count("cousin(X, Y)") = 16  (хүлээгдэж буй 16)


## Алхам 6: Рекурсив дүрэм

Хэдэн үе дээш явахыг урьдчилж мэдэхгүй тул `ancestor`-ыг суурь ба рекурсив гэсэн 2 дүрмээр бичив. Рекурсив дүрэмд `parent`-ийг эхэнд, `ancestor`-ыг төгсгөлд тавьсан. `descendant` нь `ancestor`-ыг эсрэг чиглэлд хэрэглэнэ.

In [12]:
ancestor(X, Y) <= parent(X, Y)

ancestor(X, Y) <= parent(X, Z) & ancestor(Z, Y)

descendant(X, Y) <= ancestor(Y, X);

**Шалгах:** `ancestor(X, 'temuulen')` 6 хүн, `descendant(X, 'bataa')` 9 хүн, `ancestor(X, 'anar')` нь bataa, bold, saraa, tsetseg. `ancestor` 32.

`ancestor('bataa', 'temuulen')`-ийн хэлхээ 4 хүнээс бүрдэнэ: **bataa → bold → anar → temuulen**.
1. `parent(anar, temuulen)` баримт → суурь дүрмээр `ancestor(anar, temuulen)`
2. `parent(bold, anar)` ба `ancestor(anar, temuulen)` → рекурсив дүрмээр `ancestor(bold, temuulen)`
3. `parent(bataa, bold)` ба `ancestor(bold, temuulen)` → рекурсив дүрмээр `ancestor(bataa, temuulen)`

In [13]:
print("ancestor(X, 'temuulen') :", names("ancestor(X, 'temuulen')"))
print("descendant(X, 'bataa')  :", names("descendant(X, 'bataa')"), len(names("descendant(X, 'bataa')")), 'хүн')
print("ancestor(X, 'anar')     :", names("ancestor(X, 'anar')"))
print(f'count("ancestor(X, Y)") = {count("ancestor(X, Y)")}  (хүлээгдэж буй 32)')
print()
for q in ["parent('anar', 'temuulen')", "parent('bold', 'anar')", "parent('bataa', 'bold')",
          "ancestor('bataa', 'temuulen')"]:
    print(f'{q:29} {count(q) > 0}')

ancestor(X, 'temuulen') : ['anar', 'bat', 'bataa', 'bold', 'saraa', 'tsetseg']
descendant(X, 'bataa')  : ['anar', 'bold', 'enkhee', 'munkh', 'naraa', 'oyuna', 'temuulen', 'tomor', 'zolboo'] 9 хүн
ancestor(X, 'anar')     : ['bataa', 'bold', 'saraa', 'tsetseg']
count("ancestor(X, Y)") = 32  (хүлээгдэж буй 32)

parent('anar', 'temuulen')    True
parent('bold', 'anar')        True
parent('bataa', 'bold')       True
ancestor('bataa', 'temuulen') True


## Алхам 7: Асуулт ба үгүйсгэл

`has_child`-д Y зөвхөн биед байгаа тул "ядаж нэг хүүхэдтэй" гэсэн утга. `childless`-д эхлээд `person(X)`-ээр X-д утга өгөөд дараа нь `~has_child(X)` шалгана. `great_grandparent` нь `grandparent` дээр нэг үе нэмнэ.

In [14]:
has_child(X) <= parent(X, Y)

childless(X) <= person(X) & ~has_child(X)

great_grandparent(X, Y) <= parent(X, Z) & grandparent(Z, Y);

**Шалгах:** `childless` 5 (enkhee, munkh, naraa, temuulen, zolboo). `childless('bataa')` худал, `childless('temuulen')` үнэн. `great_grandparent(X, 'temuulen')` нь bataa, saraa.

In [15]:
print('childless(X):', names('childless(X)'), count('childless(X)'))
print("childless('bataa')   :", count("childless('bataa')") > 0)
print("childless('temuulen'):", count("childless('temuulen')") > 0)
print("great_grandparent(X, 'temuulen'):", names("great_grandparent(X, 'temuulen')"))

childless(X): ['enkhee', 'munkh', 'naraa', 'temuulen', 'zolboo'] 5
childless('bataa')   : False
childless('temuulen'): True
great_grandparent(X, 'temuulen'): ['bataa', 'saraa']


**Unsafe negation ба `~parent(X, Y)`:** хоёр буруу хувилбарыг туршиж үзэв.
- `childless_unsafe`: үгүйсгэлийг `person(X)`-ээс өмнө тавьсан.
- `childless_parent`: `has_child`-ийн оронд `~parent(X, Y)`-г шууд хэрэглэсэн.

In [16]:
pyDatalog.create_terms('childless_unsafe, childless_parent')
childless_unsafe(X) <= ~has_child(X) & person(X)
childless_parent(X) <= person(X) & ~parent(X, Y)

print('childless_unsafe:', names('childless_unsafe(X)'))
print('childless_parent:', names('childless_parent(X)'))

childless_unsafe: []
childless_parent: ['enkhee', 'munkh', 'naraa', 'temuulen', 'zolboo']


- **Unsafe negation.** `~has_child(X)` эхэнд байхад X ямар ч утгагүй (холбогдоогүй) байна. Тэр үед pyDatalog "хүүхэдтэй хүн огт байхгүй юу?" гэж асуудаг болно. Хүүхэдтэй хүн байгаа тул үгүйсгэл үргэлж худал болж, алдаа ч өгөлгүй 0 хүн буцаасан. Үгүйсгэл "энэ X-д үнэн үү" гэж л шалгаж чадна, тохирох X-үүдийг өөрөө гаргаж чадахгүй. Тиймээс эхлээд `person(X)`-ээр X-д утга өгнө.
- **`~parent(X, Y)`-г яагаад шууд хэрэглэхгүй вэ.** Y зөвхөн үгүйсгэл дотор байна. Логикоор уншвал "X-ийн хүүхэд биш нэг Y байна" (∃y ¬parent(x, y)) болох ба энэ нь хүн бүрт үнэн, жишээ нь bataa нь anar-ийн эцэг эх биш. Бидэнд "X-ийн хүүхэд огт байхгүй" (¬∃y parent(x, y)) хэрэгтэй. `has_child` нь ∃y-г тусад нь бүлэглэж, `~` нь түүнийг бүхэлд нь үгүйсгэнэ. pyDatalog энэ удаад Y-г өөрөө ¬∃ гэж ойлгоод зөв 5 хүн гаргасан ч энэ нь pyDatalog-ийн онцлог. Энгийн Datalog-д үгүйсгэл доторх хувьсагч заавал эерэг нөхцөлд байх ёстой (safe negation). `has_child`-тэй хувилбарын утга ямар ч системд тодорхой.

**Асуултын хүснэгт.** Хариу бүрийг гэр бүлийн зургаас гараар бодоод программын хариутай харьцуулав. 5 ба 8-д `cousin(X, Y)`, `grandparent(X, Y)`-ийн X-ийг давхардалгүй авсан.

In [17]:
import pandas as pd

questions = [
    ('Болдын хүүхдүүд хэн бэ?', "parent('bold', X)", ['anar', 'zolboo']),
    ('Temuulen-ийн өвөө эмээ хэн бэ?', "grandparent(X, 'temuulen')", ['bold', 'tsetseg']),
    ('Munkh-ийн авга ах нар хэн бэ?', "uncle(X, 'munkh')", ['bold', 'tomor']),
    ('Enkhee-ийн үеэлүүд хэн бэ?', "cousin(X, 'enkhee')", ['anar', 'munkh', 'naraa', 'zolboo']),
    ('Үеэлтэй хүмүүс хэд вэ, хэн бэ?', 'cousin(X, Y)', ['anar', 'enkhee', 'munkh', 'naraa', 'zolboo']),
    ('Хүүхэдгүй хүмүүс хэн бэ?', 'childless(X)', ['enkhee', 'munkh', 'naraa', 'temuulen', 'zolboo']),
    ('Temuulen-ийн эцэг эхийн өвөө эмээ хэн бэ?', "great_grandparent(X, 'temuulen')", ['bataa', 'saraa']),
    ('Ач, зээтэй хүмүүс хэн бэ?', 'grandparent(X, Y)', ['bataa', 'bold', 'saraa', 'tsetseg']),
]

rows = []
for i, (question, q, by_hand) in enumerate(questions, 1):
    answer = names(q)
    rows.append({'№': i, 'Асуулт': question, 'Query': q, 'Хариулт': ', '.join(answer),
                 'Тоо': len(answer), 'Гараар таарсан': answer == by_hand})

pd.DataFrame(rows).set_index('№')

,Асуулт,Query,Хариулт,Тоо,Гараар таарсан
№,,,,,
1,Болдын хүүхдүүд хэн бэ?,"parent('bold', X)","anar, zolboo",2,True
2,Temuulen-ийн өвөө эмээ хэн бэ?,"grandparent(X, 'temuulen')","bold, tsetseg",2,True
3,Munkh-ийн авга ах нар хэн бэ?,"uncle(X, 'munkh')","bold, tomor",2,True
4,Enkhee-ийн үеэлүүд хэн бэ?,"cousin(X, 'enkhee')","anar, munkh, naraa, zolboo",4,True
5,"Үеэлтэй хүмүүс хэд вэ, хэн бэ?","cousin(X, Y)","anar, enkhee, munkh, naraa, zolboo",5,True
6,Хүүхэдгүй хүмүүс хэн бэ?,childless(X),"enkhee, munkh, naraa, temuulen, zolboo",5,True
7,Temuulen-ийн эцэг эхийн өвөө эмээ хэн бэ?,"great_grandparent(X, 'temuulen')","bataa, saraa",2,True
8,"Ач, зээтэй хүмүүс хэн бэ?","grandparent(X, Y)","bataa, bold, saraa, tsetseg",4,True


## Алхам 8: Prolog руу хөрвүүлэх

`family.pl` файлд 2–7-р алхмын мэдлэгийн санг Prolog-оор бичив. Хөрвүүлэхдээ:
- баримтын өмнөх `+`-ийг хасаж, өгүүлбэр бүрийг цэгээр төгсгөв,
- `<=` → `:-`, `&` → таслал, `~` → `\+`, `!=` → `\=`,
- нэг предикатын баримтуудыг нэг дор бичив (discontiguous анхааруулга гарахгүй),
- `has_child`-д ашиглагдахгүй Y-г `_` болгов (singleton анхааруулга гарахгүй),
- Prolog-д давталт байхгүй тул 18 `parent` баримтыг шууд бичив.

SWI-Prolog 10.1.18 дээр `family.pl`-ийг ачаалахад ямар ч анхааруулга гараагүй. Асуултуудын үр дүн:

```prolog
?- findall(P-C, parent(P, C), L), length(L, N).
L = [bataa-bold, bataa-oyuna, bataa-tomor, saraa-bold, saraa-oyuna, saraa-tomor, bold-anar, bold-zolboo, tsetseg-anar, tsetseg-zolboo, dorj-munkh, dorj-naraa, oyuna-munkh, oyuna-naraa, tomor-enkhee, ganaa-enkhee, bat-temuulen, anar-temuulen],
N = 18.

?- ancestor(bataa, temuulen).
true.

?- childless(temuulen).
true.

?- childless(bataa).
false.

?- findall(Y, sibling(oyuna, Y), L), length(L, N).
L = [bold, tomor, bold, tomor],
N = 4.

?- findall(Y, sibling(oyuna, Y), L), sort(L, S), length(S, N).
L = [bold, tomor, bold, tomor],
S = [bold, tomor],
N = 2.

?- findall(X, father(X, anar), L).
L = [bold].

?- setof(X, grandparent(X, temuulen), L).
L = [bold, tsetseg].

?- setof(X, cousin(X, enkhee), L).
L = [anar, munkh, naraa, zolboo].

?- setof(X, ancestor(X, temuulen), L), length(L, N).
L = [anar, bat, bataa, bold, saraa, tsetseg],
N = 6.

?- setof(X, childless(X), L), length(L, N).
L = [enkhee, munkh, naraa, temuulen, zolboo],
N = 5.

?- setof(X, great_grandparent(X, temuulen), L).
L = [bataa, saraa].

?- aggregate_all(count, ancestor(X, Y), N).
N = 32.

?- aggregate_all(count, cousin(X, Y), N).
N = 32.

?- aggregate_all(count, distinct(X-Y, cousin(X, Y)), N).
N = 16.
```

**Шалгах:** N = 18, `ancestor(bataa, temuulen)` true, `childless(temuulen)` true, `childless(bataa)` false. `sibling(oyuna, Y)` 4 удаа (bold, tomor хоёр хоёр удаа), `sort` хийвэл 2. Бусад асуулт pyDatalog-ийн хариутай ижил. `cousin` дээр ч Prolog 32 мөр гаргаж, давхардлыг хасвал pyDatalog-тэй адил 16 болсон.

Харьцуулахын тулд pyDatalog-оос мөн адил асуув:

In [18]:
print("pyDatalog sibling('oyuna', Y):", pyDatalog.ask("sibling('oyuna', Y)").answers, count("sibling('oyuna', Y)"))
print('pyDatalog cousin(X, Y)       :', count('cousin(X, Y)'))

pyDatalog sibling('oyuna', Y): [('bold',), ('tomor',)] 2
pyDatalog cousin(X, Y)       : 16


## Нэмэлт даалгавар 1: Нөхөр эхнэр ба хуримын ураг

Мэдлэгийн санд "нөхөр эхнэр" баримт байхгүй тул нэг хүүхдийн хоёр өөр эцэг эхийг хос гэж үзэв. `uncle_in_law` нь авга эгчийн нөхөр, `aunt_in_law` нь авга ахын эхнэр.

**Шалгах:** `spouse` 10 (5 хос, 2 чиглэлд). `uncle_in_law`: dorj нь anar, enkhee, zolboo-ийн (3 мөр). `aunt_in_law` 7 мөр: ganaa нь anar, munkh, naraa, zolboo-ийн, tsetseg нь enkhee, munkh, naraa-ийн.

In [19]:
pyDatalog.create_terms('spouse, uncle_in_law, aunt_in_law')

spouse(X, Y) <= parent(X, Q) & parent(Y, Q) & (X != Y)

uncle_in_law(X, Y) <= aunt(Z, Y) & spouse(X, Z)

aunt_in_law(X, Y) <= uncle(Z, Y) & spouse(X, Z)

print(f'count("spouse(X, Y)") = {count("spouse(X, Y)")}  (хүлээгдэж буй 10)')
print('spouse хосууд:', sorted(a for a in pyDatalog.ask('spouse(X, Y)').answers if a[0] < a[1]))
print()
print('uncle_in_law:', count('uncle_in_law(X, Y)'), 'мөр', sorted(pyDatalog.ask('uncle_in_law(X, Y)').answers))
print('aunt_in_law :', count('aunt_in_law(X, Y)'), 'мөр', sorted(pyDatalog.ask('aunt_in_law(X, Y)').answers))

count("spouse(X, Y)") = 10  (хүлээгдэж буй 10)
spouse хосууд: [('anar', 'bat'), ('bataa', 'saraa'), ('bold', 'tsetseg'), ('dorj', 'oyuna'), ('ganaa', 'tomor')]

uncle_in_law: 3 мөр [('dorj', 'anar'), ('dorj', 'enkhee'), ('dorj', 'zolboo')]
aunt_in_law : 7 мөр [('ganaa', 'anar'), ('ganaa', 'munkh'), ('ganaa', 'naraa'), ('ganaa', 'zolboo'), ('tsetseg', 'enkhee'), ('tsetseg', 'munkh'), ('tsetseg', 'naraa')]


**Яагаад `spouse` хүүхэдгүй хосыг олж чадахгүй вэ?**
- `spouse`-ийн цорын ганц нөхцөл нь нийтлэг хүүхэд. Хүүхэдгүй хос `parent` баримтгүй тул дүрэм хэзээ ч биелэхгүй.
- Closed-world assumption-оор мэдлэгийн санд байхгүй зүйлийг худал гэж үздэг тул тэд "хос биш" гэж гарна.
- Засвар: `married(X, Y)` гэсэн шинэ баримт нэмнэ, жишээ нь `+married('bataa', 'saraa')`. Тэгээд `spouse(X, Y) <= married(X, Y)`, `spouse(X, Y) <= married(Y, X)` гэж 2 чиглэлд дүрэм бичнэ. Ингэвэл хүүхэдтэй эсэхээс үл хамааран хос олдоно.

## Нэмэлт даалгавар 2: Шинэ гишүүн ба логик томъёо

dulguun-ийг 3 баримтаар нэмэв. Дүрмийг огт өөрчлөөгүй.

**Шалгах:** `sibling(X, 'temuulen')`-д dulguun гарна. `ancestor(X, 'dulguun')` нь temuulen-тэй ижил 6 хүн.

In [20]:
+male('dulguun')
+parent('bat', 'dulguun')
+parent('anar', 'dulguun')

print("sibling(X, 'temuulen'):", names("sibling(X, 'temuulen')"))
print("ancestor(X, 'dulguun') :", names("ancestor(X, 'dulguun')"))
print("ancestor(X, 'temuulen'):", names("ancestor(X, 'temuulen')"))
print('Ижил:', names("ancestor(X, 'dulguun')") == names("ancestor(X, 'temuulen')"))

sibling(X, 'temuulen'): ['dulguun']
ancestor(X, 'dulguun') : ['anar', 'bat', 'bataa', 'bold', 'saraa', 'tsetseg']
ancestor(X, 'temuulen'): ['anar', 'bat', 'bataa', 'bold', 'saraa', 'tsetseg']
Ижил: True


3 баримт нэмэхэд л `sibling`, `ancestor` зэрэг бүх дүрэм шинэ хүнд автоматаар үйлчилсэн.

**FOL томъёо:**
- father: ∀x ∀y (parent(x, y) ∧ male(x) → father(x, y))
- grandparent: ∀x ∀y ∀z (parent(x, z) ∧ parent(z, y) → grandparent(x, y)), өөрөөр ∀x ∀y (∃z (parent(x, z) ∧ parent(z, y)) → grandparent(x, y))
- ancestor (2 дүрэм хамт):
  - ∀x ∀y (parent(x, y) → ancestor(x, y))
  - ∀x ∀y ∀z (parent(x, z) ∧ ancestor(z, y) → ancestor(x, y)), өөрөөр ∀x ∀y (∃z (parent(x, z) ∧ ancestor(z, y)) → ancestor(x, y))

Толгойд байгаа x, y нь **∀**: дүрэм бүх хүний хувьд үнэн. Зөвхөн биед байгаа z нь бие дотроо **∃**: ядаж нэг дунд хүн олдоход хангалттай. Учир нь z толгойд байхгүй үед ∀z (A(z) → B) нь (∃z A(z)) → B-тэй тэнцүү. father-д биед шинэ хувьсагч байхгүй тул бүгд ∀.

**"Хүн бүр эцэг эхтэй":** ∀x (person(x) → ∃y parent(y, x))

In [21]:
print('Эцэг эхгүй хүмүүс:', [p for p in names('person(X)') if count(f"parent(X, '{p}')") == 0])

Эцэг эхгүй хүмүүс: ['bat', 'bataa', 'dorj', 'ganaa', 'saraa', 'tsetseg']


Энэ өгүүлбэр энэ мэдлэгийн санд худал. bataa нь person боловч `parent(Y, 'bataa')` баримт байхгүй. Closed-world assumption-оор санд байхгүй зүйл худал тул ∃y parent(y, bataa) худал, тэгэхээр ∀x томъёо x = bataa дээр зөрчигдөнө. Бодит амьдрал дээр bataa эцэг эхтэй ч мэдлэгийн санд бичигдээгүй. Ийм 6 хүн байна (bat, bataa, dorj, ganaa, saraa, tsetseg).

## Хариултууд

### Алхам 7: Асуултын хүснэгт

| № | Асуулт | Хариулт |
|---|---|---|
| 1 | Болдын хүүхдүүд хэн бэ? | anar, zolboo |
| 2 | Temuulen-ийн өвөө эмээ хэн бэ? | bold, tsetseg |
| 3 | Munkh-ийн авга ах нар хэн бэ? | bold, tomor |
| 4 | Enkhee-ийн үеэлүүд хэн бэ? | anar, munkh, naraa, zolboo |
| 5 | Үеэлтэй хүмүүс хэд вэ, хэн бэ? | 5 хүн: anar, enkhee, munkh, naraa, zolboo |
| 6 | Хүүхэдгүй хүмүүс хэн бэ? | enkhee, munkh, naraa, temuulen, zolboo |
| 7 | Temuulen-ийн эцэг эхийн өвөө эмээ хэн бэ? (great_grandparent) | bataa, saraa |
| 8 | Ач, зээтэй хүмүүс хэн бэ? (grandparent) | bataa, bold, saraa, tsetseg |

8 асуулт бүгд гараар бодсон хариутай таарсан. Munkh-ийн эцэг dorj-ийн ах дүү мэдлэгийн санд байхгүй тул авга ах нь зөвхөн эхийнх нь (oyuna) ах дүү bold, tomor.

### Алхам 4: `X != Y`-г хасахад нэмэгдсэн 9 нь юу вэ? Яагаад яг 9 вэ?
- Нэмэгдсэн 9 мөр бүгд sibling(X, X) хэлбэртэй, өөрөөр хэлбэл хүн өөрийнхөө ах дүү болсон.
- `X != Y` байхгүй бол `parent(P, X) & parent(P, Y)`-д X, Y хоёр нэг хүүхэд байж болно.
- Энэ нь эцэг эх нь мэдлэгийн санд байгаа хүн бүрт гарна. 15 хүнээс 6 нь (bataa, saraa, tsetseg, dorj, ganaa, bat) эцэг эхгүй тул 15 − 6 = 9.
- Нэг хүн 2 эцэг эхтэй ч pyDatalog хариуг олонлог (set) болгон хадгалдаг тул (X, X) мөр нэг л удаа тоологдоно.

### Алхам 8: pyDatalog-д давхардал яагаад гарсангүй вэ?
- Prolog нь хариу бүрийг баталгаа (proof) бүрээр нь буцаадаг. `sibling(oyuna, bold)`-ийг bataa-аар нэг, saraa-аар нэг, нийт 2 өөр замаар баталдаг тул `findall` 2 удаа цуглуулна.
- pyDatalog (Datalog) нь хариуг баримтын олонлог гэж үзнэ. Нэг баримтыг хэдэн замаар гаргасан ч нэг л удаа хадгална.
- Тиймээс `sibling(oyuna, Y)` pyDatalog-д 2, Prolog-д 4 мөр гарсан. `cousin`-д ч мөн адил 16 ба 32. Prolog-д давхардлыг `sort/2` эсвэл `setof/3`-аар арилгана.